[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C15_Classic_Architectures_Course/02_cnn_architectures/02_cnn_architectures.ipynb)

# 02 · CNN 架构演化（从零实现关键零件）

本模块不堆完整网络，而是从零实现 LeNet→ResNet 演化中**最关键的几块新零件**，并验证它们为何有效。

路线：残差块前向 → **残差让梯度直通**（朴素栈 vs 残差栈的梯度对比）→ 残差块反向(数值梯度检验) → BatchNorm(训练/推理两套统计) → 1×1 卷积(逐像素全连接) → 参数量与感受野计算 → ✏️ 练习 → 📖 答案 → 🧪 真实网络(LeNet-5/VGG-block)胶囊。

> 全程**纯 numpy、CPU**。每个前向对拍参考、每个反向过数值梯度检验。

## 0 · 公共工具：数值梯度检验 + 激活

复用全课的 `numerical_grad` / `rel_error`（模块 00 已立），再备好 ReLU 及其导数。反向的金标准就是数值梯度。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def numerical_grad(f, x, eps=1e-5):
    '''中心差分逐元素估计 df/dx。f: ndarray->标量。'''
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = x[idx]
        x[idx] = old + eps; fp = f(x)
        x[idx] = old - eps; fm = f(x)
        x[idx] = old
        g[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

def rel_error(a, b):
    return np.max(np.abs(a - b) / (np.maximum(1e-8, np.abs(a) + np.abs(b))))

def relu(x):
    return np.maximum(0.0, x)

def relu_grad(x):
    return (x > 0).astype(float)

# 自检：relu 的数值梯度 == 解析
x = rng.standard_normal((4, 5))
g_num = numerical_grad(lambda x: relu(x).sum(), x)
assert rel_error(g_num, relu_grad(x)) < 1e-6
print('✅ 工具就绪：numerical_grad / rel_error / relu')

## 1 · 残差块前向：y = F(x) + x

残差块让层只学**增量** `F(x)`，再把输入直接加回去：`y = F(x) + x`。
这里用全连接版的 `F`（两层线性 + ReLU，保持维度不变，便于 `+x`）演示——卷积版结构完全一样，只是把线性换成 conv。

In [ ]:
def residual_forward(x, W1, b1, W2, b2):
    '''残差块: F(x)=W2·relu(W1·x+b1)+b2, 返回 y=F(x)+x 以及缓存。
       x:(N,d)  W1:(d,d) W2:(d,d)  -> y:(N,d)（维度不变才能 +x）'''
    z1 = x @ W1 + b1
    a1 = relu(z1)
    Fx = a1 @ W2 + b2
    y = Fx + x                       # ★ skip connection
    cache = (x, W1, b1, W2, b2, z1, a1, Fx)
    return y, cache

N, d = 6, 8
x = rng.standard_normal((N, d))
W1 = rng.standard_normal((d, d)) * 0.1; b1 = np.zeros(d)
W2 = rng.standard_normal((d, d)) * 0.1; b2 = np.zeros(d)
y, _ = residual_forward(x, W1, b1, W2, b2)
print('x ->', x.shape, ' y ->', y.shape)
# 若把 F 学成 0（W2=0,b2=0），残差块应退化为恒等映射
y0, _ = residual_forward(x, W1, b1, np.zeros((d, d)), np.zeros(d))
assert np.allclose(y0, x), 'F=0 时残差块应 == 恒等'
print('✅ F=0 时 y==x：恒等映射是残差块的「默认值」—— 这就是 ResNet 不退化的结构保证')

## 2 · 残差为什么能堆深：梯度直通高速公路

核心论点：`y=F(x)+x` 求导得 `∂y/∂x = ∂F/∂x + I`，那个 **`+I`** 保证梯度有不衰减的通路。

我们直接做实验：堆 `L` 个块，对比**朴素栈** `x←F(x)` 与**残差栈** `x←F(x)+x` 传到输入的梯度范数。用小权重让朴素栈发生梯度消失，看残差栈如何幸免。

In [ ]:
def stack_input_grad(L, residual, d=16, scale=0.5, seed=1):
    '''堆 L 层 relu(W·x)（可选 +x），返回 sum(out) 对最初输入的梯度范数。'''
    g = np.random.default_rng(seed)
    Ws = [g.standard_normal((d, d)) * scale for _ in range(L)]
    x0 = g.standard_normal((1, d))
    # 前向，缓存每层 pre-activation
    xs = [x0]; zs = []
    x = x0
    for W in Ws:
        z = x @ W; zs.append(z)
        h = relu(z)
        x = h + x if residual else h    # ★ 残差 vs 朴素
        xs.append(x)
    # 反向：loss = sum(x_L)
    dx = np.ones_like(x)
    for i in reversed(range(L)):
        dh = dx.copy()                  # 残差: 直通项 d(out)/dx 含 +I
        dz = dh * relu_grad(zs[i])
        dx_from_F = dz @ Ws[i].T
        dx = dx_from_F + (dx if residual else 0)   # ★ +I 把梯度原样带回
    return np.linalg.norm(dx)

print(f"{'L层':>4} {'朴素栈|grad|':>16} {'残差栈|grad|':>16}")
for L in [5, 10, 20, 40]:
    gp = stack_input_grad(L, residual=False)
    gr = stack_input_grad(L, residual=True)
    print(f'{L:>4} {gp:>16.3e} {gr:>16.3e}')
# 朴素栈梯度随深度指数衰减；残差栈因 +I 保持在 O(1)
g_plain_40 = stack_input_grad(40, residual=False)
g_res_40   = stack_input_grad(40, residual=True)
assert g_res_40 > g_plain_40 * 100, '残差栈的输入梯度应远大于朴素栈'
print('\n✅ 同样 40 层：朴素栈梯度消失，残差栈靠 +I 把梯度送回浅层 —— 这就是能堆 152 层的原因')

## 3 · 残差块反向（解析）+ 数值梯度检验

把第 1 节残差块的反向**解析地**写出来，再用 `numerical_grad` 验证。
关键：`y=F(x)+x`，所以对 `x` 的梯度 = 经过 `F` 的梯度 **加上** 直通的 `dy`（即 `+I` 项）。

In [ ]:
def residual_backward(dy, cache):
    x, W1, b1, W2, b2, z1, a1, Fx = cache
    # y = Fx + x
    dFx = dy.copy()
    dx_skip = dy.copy()              # ★ 来自 +x 的直通梯度 (+I)
    # Fx = a1 @ W2 + b2
    dW2 = a1.T @ dFx
    db2 = dFx.sum(axis=0)
    da1 = dFx @ W2.T
    # a1 = relu(z1)
    dz1 = da1 * relu_grad(z1)
    # z1 = x @ W1 + b1
    dW1 = x.T @ dz1
    db1 = dz1.sum(axis=0)
    dx_F = dz1 @ W1.T
    dx = dx_F + dx_skip              # ★ F 通路 + 直通通路
    return dx, dW1, db1, dW2, db2

# 数值梯度检验（对 x 与 W1 各验一项）
x = rng.standard_normal((N, d))
W1 = rng.standard_normal((d, d)) * 0.3; b1 = rng.standard_normal(d) * 0.1
W2 = rng.standard_normal((d, d)) * 0.3; b2 = rng.standard_normal(d) * 0.1
y, cache = residual_forward(x, W1, b1, W2, b2)
dy = rng.standard_normal((N, d))    # 上游梯度
dx, dW1, db1, dW2, db2 = residual_backward(dy, cache)

loss = lambda inp: (residual_forward(inp, W1, b1, W2, b2)[0] * dy).sum()
dx_num = numerical_grad(loss, x.copy())
lossW1 = lambda Wv: (residual_forward(x, Wv, b1, W2, b2)[0] * dy).sum()
dW1_num = numerical_grad(lossW1, W1.copy())
print('dx  相对误差 =', f'{rel_error(dx,  dx_num):.2e}')
print('dW1 相对误差 =', f'{rel_error(dW1, dW1_num):.2e}')
assert rel_error(dx, dx_num) < 1e-6 and rel_error(dW1, dW1_num) < 1e-6
print('✅ 残差块反向通过数值梯度检验（含 +I 直通项）')

## 4 · BatchNorm：训练 / 推理两套统计量

前向四步：算 batch 均值方差 → 标准化 → 仿射 `γx̂+β`。**训练**用 batch 统计并滑动更新全局统计；**推理**用累积的全局统计。

下面实现一个带状态（running stats）的 BN，分别验证：训练模式输出 ~0 均值 ~1 方差；推理模式用 running stats。

In [ ]:
class BatchNorm1d:
    def __init__(self, dim, momentum=0.1, eps=1e-5):
        self.gamma = np.ones(dim); self.beta = np.zeros(dim)
        self.running_mean = np.zeros(dim); self.running_var = np.ones(dim)
        self.momentum = momentum; self.eps = eps
    def forward(self, x, training=True):
        if training:
            mu = x.mean(axis=0); var = x.var(axis=0)          # batch 统计
            # 滑动更新全局统计（供推理用）
            self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*mu
            self.running_var  = (1-self.momentum)*self.running_var  + self.momentum*var
        else:
            mu = self.running_mean; var = self.running_var      # 推理用累积统计
        xhat = (x - mu) / np.sqrt(var + self.eps)
        return self.gamma * xhat + self.beta

bn = BatchNorm1d(dim=4, momentum=0.1)
x = rng.standard_normal((100, 4)) * 5 + 3        # 非零均值、大方差
y_train = bn.forward(x, training=True)
print('训练模式输出 每特征均值 =', np.round(y_train.mean(0), 4))
print('训练模式输出 每特征方差 =', np.round(y_train.var(0), 4))
assert np.allclose(y_train.mean(0), 0, atol=1e-6), 'γ=1,β=0 时训练输出应 0 均值'
assert np.allclose(y_train.var(0), 1, atol=1e-2), '训练输出应 ~单位方差'
print('✅ 训练模式：把任意分布标准化到 0 均值 1 方差')

再喂几个 batch 让 running stats 收敛，然后验证**推理模式**用的是累积统计（不依赖当前 batch）。

In [ ]:
# 多喂几个来自同一分布的 batch，running stats 应趋近真实 (mean≈3, var≈25)
for _ in range(200):
    bn.forward(rng.standard_normal((64, 4)) * 5 + 3, training=True)
print('running_mean ≈', np.round(bn.running_mean, 2), '(真值≈3)')
print('running_var  ≈', np.round(bn.running_var, 2), '(真值≈25)')
assert np.allclose(bn.running_mean, 3, atol=0.6)
assert np.allclose(bn.running_var, 25, atol=6)

# 推理：同一个样本，无论它在哪个 batch、batch 多大，输出都应一致（用 running stats）
probe = rng.standard_normal((1, 4)) * 5 + 3
out_a = bn.forward(probe, training=False)
out_b = bn.forward(np.vstack([probe, rng.standard_normal((9, 4))]), training=False)[:1]
assert np.allclose(out_a, out_b), '推理模式输出不应依赖同 batch 的其他样本'
print('✅ 推理模式用 running stats：单样本输出确定、与 batch 无关（忘记切 train/eval 是经典 bug）')

## 5 · 1×1 卷积：逐像素的全连接

1×1 卷积对每个空间位置独立地把 `C_in` 通道线性组合成 `C_out` 通道，**不碰空间结构**。
等价于把特征图 reshape 成 `(H*W, C_in)` 做一次矩阵乘再 reshape 回去。我们实现两种写法并对拍。

In [ ]:
def conv1x1_loop(x, W, b):
    '''x:(C_in,H,W)  W:(C_out,C_in)  b:(C_out,) -> (C_out,H,W)。逐像素全连接。'''
    C_in, H, Wd = x.shape
    C_out = W.shape[0]
    y = np.zeros((C_out, H, Wd))
    for i in range(H):
        for j in range(Wd):
            y[:, i, j] = W @ x[:, i, j] + b      # 每个像素一次全连接
    return y

def conv1x1_matmul(x, W, b):
    '''向量化：reshape 成 (H*W, C_in) @ W.T 再 reshape 回。'''
    C_in, H, Wd = x.shape
    flat = x.reshape(C_in, H * Wd).T            # (H*W, C_in)
    out = flat @ W.T + b                        # (H*W, C_out)
    return out.T.reshape(W.shape[0], H, Wd)

C_in, C_out, H, Wd = 8, 3, 5, 6
x = rng.standard_normal((C_in, H, Wd))
W = rng.standard_normal((C_out, C_in)); b = rng.standard_normal(C_out)
y_loop = conv1x1_loop(x, W, b)
y_mm = conv1x1_matmul(x, W, b)
assert np.allclose(y_loop, y_mm, atol=1e-12), '两种 1x1 实现应一致'
print(f'输入 {x.shape} --1x1--> 输出 {y_mm.shape}  (通道 {C_in}->{C_out}, 空间不变)')
print('✅ 1x1 卷积 == 逐像素全连接 == reshape 后的一次矩阵乘；它在通道维降/升维')

## 6 · 参数量与感受野的计算

卷积层参数 `(k*k*C_in+1)*C_out`（与 H,W 无关——参数共享）。
感受野逐层递推：`j_l=j_{l-1}*s_l`，`r_l=r_{l-1}+(k_l-1)*j_{l-1}`，初值 `j=1,r=1`。

In [ ]:
def conv_params(k, c_in, c_out, bias=True):
    return (k * k * c_in + (1 if bias else 0)) * c_out

def receptive_field(layers):
    '''layers: [(k, s), ...]，返回最终感受野 r 与累积步幅 j。'''
    r, j = 1, 1
    for k, s in layers:
        r = r + (k - 1) * j
        j = j * s
    return r, j

# 验证：两个 3x3(stride1) 的感受野应 = 5（等价一个 5x5）
r2, _ = receptive_field([(3, 1), (3, 1)])
r3, _ = receptive_field([(3, 1), (3, 1), (3, 1)])
print('两个 3x3 感受野 =', r2, ' | 三个 3x3 感受野 =', r3)
assert r2 == 5 and r3 == 7, 'n 个 3x3 -> (2n+1)'

# 验证：两个 3x3 比一个 5x5 省参数（同 C 通道）
C = 64
p_5x5 = conv_params(5, C, C, bias=False)
p_two_3x3 = 2 * conv_params(3, C, C, bias=False)
print(f'一个 5x5: {p_5x5} 参数  vs  两个 3x3: {p_two_3x3} 参数  (省 {1-p_two_3x3/p_5x5:.0%})')
assert p_two_3x3 < p_5x5, '两个 3x3 应比一个 5x5 省参数'
print('✅ 感受野公式 + 参数量公式验证通过：小核堆叠 = 同感受野、更少参数、更多非线性')

---
## ✏️ 练习 1：瓶颈残差块的参数量

对比 256 通道下两种残差块的参数量（不计 bias 与 BN）：
- **朴素**：两个 `3×3` 卷积，256→256, 256→256
- **瓶颈**：`1×1`(256→64) + `3×3`(64→64) + `1×1`(64→256)

实现 `plain_block_params(c)` 与 `bottleneck_params(c, mid)`，复用第 6 节的 `conv_params`，返回总参数量。

In [ ]:
def plain_block_params(c):
    # TODO: 两个 3x3 卷积 c->c 的参数和（bias=False）
    raise NotImplementedError

def bottleneck_params(c, mid):
    # TODO: 1x1(c->mid) + 3x3(mid->mid) + 1x1(mid->c) 的参数和（bias=False）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
pp = plain_block_params(256)
bp = bottleneck_params(256, 64)
print(f'朴素块 = {pp} 参数 | 瓶颈块 = {bp} 参数 | 省 {pp/bp:.1f}x')
assert pp == 2 * (3*3*256*256)
assert bp == (1*1*256*64) + (3*3*64*64) + (1*1*64*256)
assert bp < pp / 5, '瓶颈块应省下数倍参数'
print('✅ 练习 1 通过：瓶颈块用 1x1 降维省下十几倍参数')

## ✏️ 练习 2：VGG 块的感受野

一个 VGG stage 常是「`3×3` conv ×2 + `2×2` maxpool(stride 2)」。

用第 6 节的 `receptive_field` 计算**两个这样的 stage 串起来**之后的感受野。
提示：池化也算一层 `(k=2, s=2)`。实现 `vgg_two_stage_rf()` 返回最终感受野 r。

In [ ]:
def vgg_two_stage_rf():
    # TODO: 构造 layers = [3x3 s1, 3x3 s1, 2x2 s2,  3x3 s1, 3x3 s1, 2x2 s2]
    #       调用 receptive_field 返回 r
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
r = vgg_two_stage_rf()
# 手算核对：r=1;(+2)=3;(+2)=5; pool(+1*1)=6,j=2; (+2*2)=10;(+4)=14; pool(+1*2)=16
print('两个 VGG stage 的感受野 r =', r)
assert r == 16, '逐层递推应得 16'
print('✅ 练习 2 通过：会算多层（含池化）累积感受野')

## ✏️ 练习 3：BatchNorm 反向（数值梯度检验）

实现 BN 前向的**纯函数版** `bn_forward(x, gamma, beta, eps)`（训练模式，返回标量 loss 便于检验），
再实现 `bn_backward` 给出对 `x, gamma, beta` 的梯度，并用 `numerical_grad` 验证 `dgamma`。

提示：先写 `loss = sum(γ*x̂+β)`，可只解析地求 `dgamma = sum(x̂)`、`dbeta = sum(1)`，`dx` 可暂用数值梯度占位。

In [ ]:
# 给定的辅助：标准化（不含仿射），供本题复用
def bn_normalize(x, eps=1e-5):
    mu = x.mean(0); var = x.var(0)
    return (x - mu) / np.sqrt(var + eps)

def bn_dgamma(x, dout, eps=1e-5):
    # TODO: y=γ*x̂+β，则 dgamma = sum(dout * x̂, axis=0)。x̂ 用 bn_normalize(x)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
def bn_normalize(x, eps=1e-5):                 # 给定辅助（自测自带，确保可独立运行）
    mu = x.mean(0); var = x.var(0)
    return (x - mu) / np.sqrt(var + eps)
x = rng.standard_normal((20, 4)) * 2 + 1
gamma = rng.standard_normal(4); beta = rng.standard_normal(4)
dout = rng.standard_normal((20, 4))
dg = bn_dgamma(x, dout)
# 数值检验：loss = sum(dout * (γ*x̂+β))，对 gamma 求梯度
def lossg(g):
    return (dout * (g * bn_normalize(x) + beta)).sum()
dg_num = numerical_grad(lossg, gamma.copy())
print('dgamma 相对误差 =', f'{rel_error(dg, dg_num):.2e}')
assert rel_error(dg, dg_num) < 1e-5
print('✅ 练习 3 通过：BN 对 γ 的梯度 = sum(dout * x̂)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def plain_block_params(c):
    return 2 * conv_params(3, c, c, bias=False)

def bottleneck_params(c, mid):
    return (conv_params(1, c, mid, bias=False)
            + conv_params(3, mid, mid, bias=False)
            + conv_params(1, mid, c, bias=False))

In [ ]:
# 练习 2 参考答案
def vgg_two_stage_rf():
    layers = [(3, 1), (3, 1), (2, 2),  (3, 1), (3, 1), (2, 2)]
    return receptive_field(layers)[0]

In [ ]:
# 练习 3 参考答案
def bn_dgamma(x, dout, eps=1e-5):
    return (dout * bn_normalize(x, eps)).sum(axis=0)

---
## 🧪 真实数据胶囊：真实网络的参数量与感受野

用**真实网络规格**算两笔账，把公式接到现实：
① **LeNet-5**（LeCun 1998）的卷积层参数量；② **VGG-16** 第一个 stage 的感受野。

再用 optdigits（8×8 真实手写数字）跑一遍 1×1 卷积，确认它在真实数据上改变通道数。

In [ ]:
# LeNet-5 真实规格（经典版）：
#  C1: 6 个 5x5 卷积，输入 1 通道
#  C3: 16 个 5x5 卷积，输入 6 通道
lenet_convs = [
    ('C1', 5, 1, 6),
    ('C3', 5, 6, 16),
]
total = 0
for name, k, cin, cout in lenet_convs:
    n = conv_params(k, cin, cout, bias=True)
    total += n
    print(f'{name}: {k}x{k} {cin}->{cout}  = {n} 参数')
print(f'LeNet-5 两卷积层共 {total} 参数（注意：相比末端全连接极少 —— 参数共享的威力）')
assert conv_params(5, 1, 6) == (5*5*1+1)*6 == 156    # 经典 LeNet C1 = 156
print('✅ C1 = 156 参数，与 LeNet-5 原论文一致')

In [ ]:
# VGG-16 第一个 stage: conv3-64, conv3-64, maxpool —— 算其感受野
vgg_stage1 = [(3, 1), (3, 1), (2, 2)]
r, j = receptive_field(vgg_stage1)
print(f'VGG stage1 后 感受野={r}, 累积步幅={j}')
assert (r, j) == (6, 2)
# 在 optdigits 上跑 1x1 卷积：把单通道 8x8 数字「升」成 4 通道特征
from sklearn.datasets import load_digits
img = load_digits().images[0][None, :, :]      # (1,8,8) 单通道
W = rng.standard_normal((4, 1)); b = rng.standard_normal(4)
feat = conv1x1_matmul(img, W, b)
print('optdigits 单数字 (1,8,8) --1x1--> ', feat.shape, '(升到 4 通道，空间不变)')
assert feat.shape == (4, 8, 8)
print('✅ 真实网络规格与真实数据上的计算全部吻合')

**🧪 胶囊练习**：实现 `alexnet_first_layer_rf()`——AlexNet 第一层是 `11×11` 卷积、stride 4。
算它**单独一层**的感受野（应等于核大小 11）。

In [ ]:
def alexnet_first_layer_rf():
    # TODO: 用 receptive_field([(11, 4)]) 返回 r
    raise NotImplementedError

In [ ]:
# 自测
r = alexnet_first_layer_rf()
assert r == 11, '单层感受野 = 核大小'
print(f'AlexNet 首层 11x11 s4 的感受野 = {r}')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def alexnet_first_layer_rf():
    return receptive_field([(11, 4)])[0]

### 小结
- CNN 演化 = 把网络做深而不崩的历史：ReLU/dropout(AlexNet) → 小核堆叠(VGG) → 残差+BN(ResNet)。
- **残差连接** `F(x)+x`：恒等是默认值（不退化）；`+I` 让梯度直通（能堆上百层）。
- **BatchNorm**：训练用 batch 统计 + 滑动更新；推理用 running stats —— 切错模式是经典 bug。
- **1×1 卷积** = 逐像素全连接，管通道升降维；瓶颈块靠它省十几倍算力。
- 参数量 `(k²C_in+1)C_out`（与 H,W 无关）；感受野逐层递推 `r+=( k-1)·j, j*=s`。

> 划重点：**残差连接与归一化，下一步会在 Transformer（C1）里原样重现**。

下一站：**模块 03 · RNN 与 BPTT** —— 从空间转向时间，参数共享搬到时间维。